In this notebook, we will go through the key elements to make the adata object in downstream single cell analysis.

- Nuclear segmentation (TODO)
- Spatial coordinates (TODO)

- MERFISH decoding

- ecDNA cts
- dispersion score

- MALAT/malat quantification

# Nuclear segmentation

# Spatial coordinates

# MERFISH decoding

In [ ]:
set_ = ''
segm_folder = r'\\192.168.0.34\Papaya5\Brett\Segmentation'
dic_ = np.load(segm_folder +r'\imfsegmV2--'+set_+'.npz')
coordsf,resc,segm_fls,pix_size = dic_['coordsf'],dic_['resc'],dic_['segm_fls'],dic_['pix_size']
imfsegm = dic_['imfsegm']
fovs = [os.path.basename(fl).split('--')[0]for fl in segm_fls]

In [ ]:
def get_best_translation_pointsT(fl,save_folder,fl_ref,save_folder_ref,set_='',resc=5,
                                 psf_fl = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2-60x1p5_final.npy',
                                im_med_fl = r'C:\Scripts\NMERFISH\flat_field\Scope2_60X1p_med_col_raw3.npz'):
    obj = get_dapi_features(fl,save_folder,set_,psf_fl=psf_fl,im_med_fl=im_med_fl)
    obj_ref = get_dapi_features(fl_ref,save_folder_ref,set_,psf_fl=psf_fl,im_med_fl=im_med_fl)
    tzxyf,tzxy_plus,tzxy_minus,N_plus,N_minus = np.array([0,0,0]),np.array([0,0,0]),np.array([0,0,0]),0,0
    if (len(obj.Xh_plus)>0) and (len(obj_ref.Xh_plus)>0):
        X = obj.Xh_plus[:,:3]
        X_ref = obj_ref.Xh_plus[:,:3]
        tzxy_plus,N_plus = get_best_translation_points(X,X_ref,resc=resc,return_counts=True)
    if (len(obj.Xh_minus)>0) and (len(obj_ref.Xh_minus)>0):
        X = obj.Xh_minus[:,:3]
        X_ref = obj_ref.Xh_minus[:,:3]
        tzxy_minus,N_minus = get_best_translation_points(X,X_ref,resc=resc,return_counts=True)
    if np.max(np.abs(tzxy_minus-tzxy_plus))<=2:
        tzxyf = -(tzxy_plus*N_plus+tzxy_minus*N_minus)/(N_plus+N_minus)
    else:
        tzxyf = -[tzxy_plus,tzxy_minus][np.argmax([N_plus,N_minus])]
    return [tzxyf,tzxy_plus,tzxy_minus,N_plus,N_minus]
def main_analysisV1(fov,imfsegm,fovs,coordsf,set_='',
                    save_folder =r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ',
                   save_folder_ref = r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ',
                   save_fl_ref = r'\\192.168.0.34\Papaya5\Brett\final_spots\scoresRefQ.pkl',
                   save_folder_all = r'\\192.168.0.34\Papaya5\Brett\final_spots',
                   segm_tag = 'Q1_MER',redo=False,resc=resc.astype(int),
                   drift_semgentation = None,thcut=-1,pix_size=pix_size):


    if not os.path.exists(save_folder_all): os.makedirs(save_folder_all)
    save_fl = save_folder_all+os.sep+'saveDecodedFinalV1--'+os.path.basename(save_folder)+'--'+fov.split('.')[0]+'--'+set_+'.npz'
    dec = None
    if not os.path.exists(save_fl) or redo:
        ####### Load in the decoded spot data #############################
        dec = decoder_simple(save_folder,fov,set_)
        dec.ncols = 3
        dec.load_decoded()
        dec.dist_best=np.load(dec.decoded_fl)['dist_best']  ######## new
        #scoresRefT = get_score_per_color(dec,include_dbits=True) #### new
        scoresRefT = np.load(save_fl_ref,allow_pickle=True)
        get_score_withRef(dec,scoresRefT,plt_val=True,gene=None,iSs = None,th_min=-10,include_dbits=True) ### new
        print("Molecules size:",dec.XH_pruned.shape)
        print("Covering reaoduts:",np.unique(dec.XH_pruned[:,:,-1]))
        dec.th=thcut
        plot_statistics(dec)

                ####### Threhold the decoded spot data #############################
        keepf =  dec.scoreA>dec.th ### keep good score
        icodesf = dec.icodesN[keepf]
        XHfpr = dec.XH_pruned[keepf]

        ### Reorganize data  ###############################
        XHf = np.nanmean(XHfpr,axis=1)
        XF = XHf[:,[0,1,2,-5,-4,-3,-2,-1,-1,-1,-1]]
        #zc,xc,yc,bk-7,a-6,habs-5,hn-4,h-3
        XF[:,-1] = dec.scoreA[keepf]
        XF[:,-2] = np.where(keepf)[0]
        mnD = np.nanmean(np.linalg.norm((XHf[:,np.newaxis]-XHfpr)[:,:,:3],axis=-1),axis=-1)
        XF[:,-3]=mnD
        mnH = np.nanmean(np.abs((XHf[:,np.newaxis]-XHfpr)[:,:,-3]),axis=-1)
        XF[:,-4]=mnH
        genesf = dec.gns_names[icodesf]

        ### Apply drift to match the segmentation reference ########################
        def get_drft_dic(fl_all_drift):
            drft,flds,fov_,fl_ref= np.load(fl_all_drift,allow_pickle=True)
            flds = [os.path.dirname(fld) if os.path.basename(fld)=='' else fld for fld in flds]
            return {('--'.join(fld.split(os.sep)[-2:]) if '--' in os.path.basename(fld) else os.path.basename(fld)):drft_[0] 
                    for fld,drft_ in zip(flds,drft)}

        #fl_all_drift = drift_folder+os.sep+'driftNew_'+dec.fov+'--'+dec.set_+'.pkl'
        #fl_all_drift = save_folder+os.sep+'driftNew_'+dec.fov+'--'+dec.set_+'.pkl'
        #drft,flds,fov_,fl_ref= np.load(fl_all_drift,allow_pickle=True)
        #drft_dic_all = get_drft_dic(fl_all_drift)

        fl_all_drift = save_folder+os.sep+'driftNew_'+dec.fov+'--'+dec.set_+'.pkl'
        drft,flds,fov_,fl_ref= np.load(fl_all_drift,allow_pickle=True)

        
        #### bring to space of the segmentation
        drft_dic_group = get_drft_dic(fl_all_drift)
        drft_dic_all = drft_dic_group
        if segm_tag not in drft_dic_all:
            resT = get_best_translation_pointsT(fl_ref,save_folder,segm_tag+os.sep+os.path.basename(fl_ref),save_folder_ref,set_=set_,resc=5)
            drft_dic_all = {}
            print("Calculated drift to segmentation:",resT[0])
            drft_dic_all[os.path.basename(os.path.dirname(fl_ref))]=resT[0]
            drft_dic_all[segm_tag]=np.array([0,0,0])
        
        
        key_segm = [key for key in drft_dic_all if segm_tag in key][0]
        key_align = np.intersect1d(list(drft_dic_all.keys()),list(drft_dic_group.keys()))[0]
        if drift_semgentation is None:
            drift_semgentation = drft_dic_all[key_segm]
        print(drift_semgentation)
        th1_to_h0 = drift_semgentation-drft_dic_all[key_align]
        th1_gr = drft_dic_group[key_align]
        tzxyf = th1_gr+th1_to_h0

        XF[:,:3]=XF[:,:3]-tzxyf #### do not add here

        
                
        ### Load in the current fov segmentation ######################################
        fovs = list(fovs)
        ifov = fovs.index(dec.fov)
        coords= coordsf[ifov]
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]
        min_ = np.nanmin(XF[:,:3],axis=0)
        #max_ = np.nanmax(XF[:,:3],axis=0)

        xIm,yIm = np.round([xIm,yIm]+(min_/resc)[1:]).astype(int)
        ### deal with the case of outside image
        Im = np.array([xIm,yIm])
        min_[1:]=min_[1:]-Im*(Im<0)*resc[1:]
        ### recenter
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]
        xIm,yIm = np.round([xIm,yIm]+(min_/resc)[1:]).astype(int)
        XF__=XF[:,:3]-min_
        max_ = np.nanmax(XF__[:,:3],axis=0)
        xIM,yIM = np.round([xIm,yIm]+(max_/resc)[1:]).astype(int)
        xIm=np.max([0,xIm])
        yIm=np.max([0,yIm])
        im_segm_ = np.array(imfsegm[:,xIm:xIM,yIm:yIM])
        dec.imfsegm0 = im_segm_.copy()
        print(xIm,xIM,yIm,yIM)
        dec.im_segm_ = expand_segmentation(im_segm_,int(9*1.5))
        ###keepO
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]#### removeBB
        dec.im_segm_O = np.array(imfsegm[:,xIm:xIM,yIm:yIM])    
        
        dec.XF__ = XF__
        dec.XF = XF
        dec.resc = resc

        #### get cell ids #######################
        def get_cell_id(dec,Xh,resc=resc,plot_val = False):
            if not hasattr(dec,'im_segm_'): 
                good = np.ones(len(Xh),dtype=bool)
                cells_ = np.zeros(len(Xh),dtype=int)
                return cells_,good
            im_segm = dec.im_segm_
            dec.shapesm = dec.im_segm_.shape
            Xcms = Xh[:,:3]
            Xred = np.round(Xcms/resc).astype(int)
            good = ~np.any((Xred>=dec.shapesm)|(Xred<0),axis=-1)
            Xred = Xred[good]
            dec.Xred =Xred
            if plot_val:
                v = napari.view_points(dec.Xred ,size=1.5)
                v.add_labels(dec.im_segm_)
            return im_segm[tuple(Xred.T)],good


        cell_id,good =get_cell_id(dec,XF__,resc=resc,plot_val = False)
        #XF_ = np.concatenate([XF[good],cell_id[:,np.newaxis]],axis=-1)
        genesf_ = genesf[good]
        dec.genesf_ = genesf_
        dec.genesf = genesf
        iset = 0#int(dec.set_.split('_set')[-1])
        ifov = int(dec.fov.split('_')[-1].split('.')[0])
        isets = np.array([iset]*len(cell_id))[:,np.newaxis]
        ifovs = np.array([ifov]*len(cell_id))[:,np.newaxis]
        cell_id = cell_id[:,np.newaxis]
        XF_ = np.concatenate([XF[good],cell_id,ifovs,isets],axis=-1)

        #get_xyfov(dec)
        ### add absolute positiuons of molecules in um
        pixel_size = np.array([0.5,0.10833,0.10833])*resc
        XF_ = XF_[:,list(np.arange(XF_.shape[-1]))+[-1,-1,-1,-1]]
        in_nuc = (dec.imfsegm0[tuple(dec.Xred.T)]>0).astype(np.float32)
        XF_[:,-4]=in_nuc
        XF_[:,-3:]=((XF_[:,:3]/resc)+[0,coords[0],coords[2]])*pixel_size
        
        header = ['z','x','y','abs_brightness','cor','brightness','color','mean_bightness_variation','mean_distance_variation',
                  'index_from_XH_pruned','score','cell_id','ifov','iset','in_nuc','z_um_abs','x_um_abs','y_um_abs']



        ##### get info for cells #########################
        if not hasattr(dec,'im_segm_'): dec.im_segm_=np.zeros(np.array([30,3000,3000])//2)
        icells,vols = np.unique(dec.im_segm_,return_counts=True)
        cms = np.array(ndimage.center_of_mass(np.ones_like(dec.im_segm_),dec.im_segm_,icells))
        #icells,vols = np.unique(dec.im_segm_,return_counts=True)
        cms = np.array(ndimage.center_of_mass(np.ones_like(dec.im_segm_),dec.im_segm_,icells))

        cells_abs_pos = (cms+[0,coords[0],coords[2]])*pixel_size

        cellinfo = cms[:,[0,0,0,1,2,0,0,0]]
        cellinfo[:,0]=icells
        cellinfo[:,1]=vols
        cellinfo[:,-3:]=cells_abs_pos

        header_cells = ['cell_id','volm','zc','xc','yc','z_um_abs','x_um_abs','y_um_abs']
        plt.close('all')

        if True:
            np.savez_compressed(save_fl,XF=XF_,
                                genes = genesf_,cellinfo=cellinfo,header_cells=header_cells,header=header)
        print("Finished",save_fl)
    return dec
def get_score_per_color(dec,include_dbits=False):
    H = np.nanmedian(dec.XH_pruned[...,-3],axis=1)
    D = dec.XH_pruned[...,:3]-np.nanmean(dec.XH_pruned[...,:3],axis=1)[:,np.newaxis]
    D = np.nanmean(np.linalg.norm(D,axis=-1),axis=-1)
    if include_dbits:
        db = dec.dist_best
        score = np.array([H,-D,-db]).T
    else:
        score = np.array([H,-D]).T
    score = np.sort(score,axis=0)
    return [score[dec.XH_pruned[:,0,-2]==icol] for icol in np.arange(dec.ncols)]
def get_score_withRef(dec,scoresRef,plt_val=False,gene=None,iSs=None,th_min=-np.inf,include_dbits=False):
    H = np.nanmedian(dec.XH_pruned[...,-3],axis=1)
    D = dec.XH_pruned[...,:3]-np.nanmean(dec.XH_pruned[...,:3],axis=1)[:,np.newaxis]
    D = np.nanmean(np.linalg.norm(D,axis=-1),axis=-1)
    if include_dbits:
        db = dec.dist_best
        score = np.array([H,-D,-db]).T
    else:
        score = np.array([H,-D]).T
    
    keep_color = [np.nanmedian(dec.XH_pruned[:,:,-2],axis=1)==icol for icol in np.arange(dec.ncols)]
    scoreA = np.zeros(len(H))
    for icol in range(dec.ncols):
        scoresRef_ = scoresRef[icol]
        score_ = score[keep_color[icol]]
        from scipy.spatial import KDTree
        scoreA_ = np.zeros(len(score_))
        if iSs is None: iSs = np.arange(scoresRef_.shape[-1])
        for iS in iSs:
            dist_,inds_ = KDTree(scoresRef_[:,[iS]]).query(score_[:,[iS]])
            scoreA_+=np.log((inds_+1))-np.log(len(scoresRef_))
        scoreA[keep_color[icol]]=scoreA_
    dec.scoreA =scoreA
    if plt_val:
        bad_igns = [ign for ign,gn in enumerate(dec.gns_names) if 'blank' in gn.lower()]
        good_igns = [ign for ign,gn in enumerate(dec.gns_names) if 'blank' not in gn.lower()]
        is_good_gn = np.in1d(dec.icodesN,good_igns)
        
        plt.figure()
        kp = scoreA>th_min
        plt.hist(scoreA[(is_good_gn)&kp],density=True,bins=100,alpha=0.5,label='all genes')
        if gene is not None:
            is_gn = dec.icodesN==(list(dec.gns_names).index(gene))
            plt.hist(scoreA[(is_gn)&kp],density=True,bins=100,alpha=0.5,label=gene)
        plt.hist(scoreA[(~is_good_gn)&kp],density=True,bins=100,alpha=0.5,label='blanks');
        plt.legend()

In [ ]:
## mouse
dec = main_analysisV1(fov,imfsegm,fovs,coordsf,set_='',
                    save_folder =r'\\192.168.0.34\Papaya5\Brett\MERFISH_Analysis',
                   save_folder_ref = r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ',
                   save_fl_ref = r'\\192.168.0.34\Papaya5\Brett\final_spots\scoresRef.pkl',
                   save_folder_all = r'\\192.168.0.34\Papaya5\Brett\final_spotsV2',
                   segm_tag = 'Q1_MER',redo=False)

In [ ]:
## human 
dec2 = main_analysisV1(fov,imfsegm,fovs,coordsf,set_='',
                    save_folder =r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ',
                   save_folder_ref = r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ',
                   save_fl_ref = r'\\192.168.0.34\Papaya5\Brett\final_spots\scoresRefQ.pkl',
                   save_folder_all = r'\\192.168.0.34\Papaya5\Brett\final_spotsV2',
                   segm_tag = 'Q1_MER',redo=False)

## format cell x gene matrices

In [ ]:
save_fls_ = glob.glob(r'\\192.168.0.34\Papaya5\Brett\final_spotsV2\saveDecoded*')

In [ ]:
tags_lib = ['-M1','-H1']
fls_lib = [r'C:\Scripts\NMERFISH\codebooks\codebook_0_New_DCBB-300_MERFISH_encoding_2_21_2023.csv',
           r'C:\Scripts\NMERFISH\codebooks\Roy_Qhybes_huamnExtra__2_2_2024_withBlank.csv']

namesf = []
namesflib = []
dic_bit_lib = {}
for fl_l,tag_l in zip(fls_lib,tags_lib):
    dfL = pd.read_csv(fl_l)
    #namesf+=[nm+tag_l if 'blank' in nm else nm for nm in dfL['name']] #### changed
    namesf+=[nm+'-'+tag_l for nm in dfL['name']]
    for bit in dfL.columns[2:]: 
        for gn in dfL['name'][dfL[bit]==1]:
            if 'blank' not in gn:
                dic_bit_lib[gn]=dic_bit_lib.get(gn,[])+[tag_l.zfill(3)+'__'+bit]
    namesflib+=[nm+'--'+tag_l for nm in dfL['name']]
genesf = np.unique(namesf)
dic_nmlib_index = dict(zip(genesf,np.arange(len(genesf))))

dic_libs = {}
for gn in namesflib:
    gn_,lib_ = gn.split('---')[0],gn.split('---')[-1]
    if gn_ not in dic_libs: dic_libs[gn_]=[]
    dic_libs[gn_].append(lib_)
#dic_bit_lib
dic_bit_lib_inv = {}
for gn in dic_bit_lib:
    for bit in dic_bit_lib[gn]:
        dic_bit_lib_inv[bit]=dic_bit_lib_inv.get(bit,[])+[gn]
dic_bit_lib_invf = {int(key.split('bit')[-1]):dic_bit_lib_inv[key] for key in dic_bit_lib_inv}

In [ ]:
#dic_cells = {}
#dic_cells_alt={}
vol_th = 5000
cells_all = None
keys = []
for save_fl in tqdm(save_fls_[:]):
    #save_fl = save_fls_[0]
    dic = np.load(save_fl)
    cells = dic['cellinfo'][1:]
    header_cells = list(dic['header_cells'])
   
    vols = cells[:,header_cells.index('volm')]

    icellsf = cells[:,0].astype(int)
    vols = cells[:,header_cells.index('volm')]
    
    vol_th_ = vols>vol_th
    icellsf = icellsf[vol_th_]
    cells = cells[vol_th_]
    base = os.path.basename(save_fl).split('--')[-2]+'_'
    keys.extend([base+str(int(cell)) for cell in cells[:,0]])
    cells_all = cells if cells_all is None else np.concatenate([cells_all,cells],axis=0)

    
volms = cells_all[:,1]
index = np.argsort(volms)[::1]
keys_ = np.array(keys)[index]
cells_all_=cells_all[index]
dice_cells = dict(zip(keys_,cells_all_))
keysF = np.array(list(dice_cells.keys()))
Xcells = np.array([dice_cells[cell][[0,-3,-2,-1]]for cell in keysF])

tree = cKDTree(Xcells)
res = tree.query_ball_tree(tree,10)
res_ = []
indCells = Xcells[:,0]

for ir,r in enumerate(tqdm(res)):
    ind_ = indCells[r]
    res_.append(np.array(r)[ind_==indCells[ir]])
dic_match={}
for ir,r in enumerate(res_):
    for r_ in r:
        dic_match[keysF[ir]]=keysF[r_]


In [ ]:
#df will be of size genesf and cellidsf
#cell_df,df=None,None
fov_bit_problems={}
dfF = None
cell_dfF = None
counter=0
dic_cells = {}
cellidsf = []
df = np.zeros([0,len(genesf)])
set_ = ''
plate_dic = {'MERFISH_Analysis':'--M1','MERFISH_AnalysisQZ':'--M2','MERFISH_AnalysisQ':'--H1'}
for save_fl in tqdm(save_fls_[:]):
    #save_fl = save_fls_[0]
    dic = np.load(save_fl)
    XF = dic['XF']
    header = list(dic['header'])
    genes_decoded = dic['genes']
    cells = dic['cellinfo'][1:]
    header_cells = list(dic['header_cells'])
    ### Rethrehold based on score
    iscore = list(header).index('score')
    scoreA = XF[:,iscore]
    keep = scoreA>-1.5



    icell = header.index('cell_id')

    icells = XF[keep,icell].astype(int)
    vols = cells[:,header_cells.index('volm')]

    icellsf = cells[:,0].astype(int)
    vols = cells[:,header_cells.index('volm')]
    
    vol_th_ = vols>vol_th
    icellsf = icellsf[vol_th_]
    cells = cells[vol_th_]
    
    
    
    
    keep_vol = np.in1d(icells,icellsf) ### is molecule part of a big enough cell
    icells=icells[keep_vol] #final cell indexes of the molecules
    
    base_cell = os.path.basename(save_fl).split('--')[-2]+'_'
    icells_match = np.array([dic_match[base_cell+str(int(icell))]for icell in icells])
    
    ugns,gns_i = np.unique(genes_decoded[keep][keep_vol],return_inverse=True)
    
    
    
    icells_unique,icells_inv = np.unique(icells_match,return_inverse=True)
    #icells_unique = list(icells_unique)+list(np.setdiff1d(icellsf,icells_unique))### add back in the 0 count cells
    #icells_unique = np.array(icells_unique)
    mcell = len(icells_unique)#+1
    gn_cell,gn_cell_cts = np.unique(mcell*gns_i+icells_inv,return_counts=True)
    igns_ = gn_cell//mcell
    icells_ = gn_cell%mcell
    Mcts = np.zeros([mcell,len(ugns)])
    Mcts[icells_,igns_]=gn_cell_cts
    
    _,plate,fov,set_ = os.path.basename(save_fl).split('--')
    plate = plate_dic.get(plate,'L1')
    ugns_ = [gn+plate for gn in ugns]
    igns = [dic_nmlib_index[gn] for gn in ugns_]
    
    #icells_unique
    in_common_cells = np.in1d(icells_unique,cellidsf)#,assume_unique=True)
    cellidsf_dic = dict(zip(cellidsf,np.arange(len(cellidsf))))
    in_common_cells_f = np.array([cellidsf_dic[cell_] for cell_ in icells_unique[in_common_cells]],dtype=int)
    #in_common_cells_f = np.in1d(cellidsf,icells_unique)#,assume_unique=True)
    vals = np.zeros([np.sum(in_common_cells),len(genesf)])
    prev_vals = df[in_common_cells_f][:,igns]
    vals[:,igns] = np.max([Mcts[in_common_cells],prev_vals],axis=0)

    ibad_genes = []
    fov__ = save_fl.split('--')[-2]
    if fov__ in fov_bit_problems:####################################
        bad_bits = fov_bit_problems[fov__]
        bad_genes = [gn for bit in bad_bits for gn in dic_bit_lib_invf[bit]]
        genesf_ = list(genesf)
        ibad_genes = [genesf_.index(gn) for gn in bad_genes]
    ### modify vals to add nan
    if len(ibad_genes)>0:
        vals[:,ibad_genes]=np.nan
    if np.sum(in_common_cells_f)>0:
        df[in_common_cells_f] = np.nanmax([vals,df[in_common_cells_f]],axis=0)
    
    #df[in_common_cells_f][:,igns] = np.max([df[in_common_cells_f][:,igns],Mcts[in_common_cells]],axis=0)
    not_in_common_cells = ~in_common_cells
    extra_df = np.zeros([np.sum(not_in_common_cells),len(genesf)])
    extra_df[:,igns]=Mcts[not_in_common_cells]
    ### modify extra_df to add nan, genesf on the last dim
    if len(ibad_genes)>0:
        extra_df[:,ibad_genes]=np.nan
    df = np.concatenate([df,extra_df])
    cellidsf+=list(icells_unique[not_in_common_cells])

In [ ]:
dff = pd.DataFrame(df,columns=genesf,index=cellidsf)
cell_df = np.array([dice_cells.get(cell,[0]*len(header_cells)) for cell in cellidsf])
cell_df = pd.DataFrame(cell_df,columns=header_cells,index=cellidsf)

In [ ]:
data_folder =   r'\\192.168.0.34\Papaya5\Brett\DataFramesV2'
if not os.path.exists(data_folder): os.makedirs(data_folder)
set_ = ''
dff.to_pickle(data_folder+r'\dff'+set_+'_1p5thV2.pkl') ### cell x gene
cell_df.to_pickle(data_folder+r'\cell_df'+set_+'_1p5thV2.pkl') ## cell metadata

## assemble it to adata

In [ ]:
data_folder =   r'\\192.168.0.34\Papaya5\Brett\DataFramesV2'
set_=''
dff=pd.read_pickle(data_folder+r'\dff'+set_+'_1p5thV2.pkl')
cell_df=pd.read_pickle(data_folder+r'\cell_df'+set_+'_1p5thV2.pkl')

In [ ]:
genes = [gn for gn in dff.columns if '_smFISH' not in gn and 'blank' not in gn]
dfR_ = dff[genes].copy()
dfR_ = dfR_.replace(np.nan, 0)
cell_dfR_ = cell_df

scdata2 = sc.AnnData(dfR_)
xcells = np.array(cell_df['x_um_abs'])
ycells = np.array(cell_df['y_um_abs'])
Xcells = np.array([xcells,ycells]).T
scdata2.obsm["X_spatial"] = Xcells
scdata2.obsm["X_raw"] = scdata2.X.copy()
scdata2.obs['volm']=cell_dfR_['volm']
sc.pp.calculate_qc_metrics(scdata2, percent_top=None, inplace=True)
sc.pp.normalize_total(scdata2, target_sum=np.median(scdata2.obs["total_counts"]))
sc.pp.log1p(scdata2)

In [ ]:
sc.pp.neighbors(scdata2)  #metric='correlation'
sc.tl.umap(scdata2,random_state=9,min_dist=0.2)
sc.tl.leiden(scdata2, resolution=1.5) #### 

In [ ]:
artefacts = [24,31,32,34,35]
leiden = scdata2.obs['leiden'].astype(int)
keep = ~np.in1d(leiden,artefacts)
scdata2f = scdata2[keep]

In [ ]:
scdata2f.write_h5ad(data_folder+os.sep+'scdata_1p5_Brett_6_19_2024.h5ad')

# DNA counts

## spot fitting

In [ ]:
all_flds = [r'\\192.168.0.113\Papaya9\Brett\1Hyb_P1F1F2F3']
save_folder = r'\\192.168.0.34\Papaya5\Brett\DNA_analysis'
if not os.path.exists(save_folder): os.makedirs(save_folder)
set_ = ''
#fov = 'Conv_zscan__200'
for fov in fovs:
    compute_fits(save_folder,fov,all_flds,redo=False,ncols=4,
                    psf_file = psf_file,try_mode=True,old_method=False)
    compute_drift_features(save_folder,fov,all_flds,set_,redo=False,gpu=True)

## get DNA counts and compute dispersion score

In [ ]:
from ioMicro_old_2_2_2024 import *
def get_best_translation_pointsT(fl,save_folder,fl_ref,save_folder_ref,set_='',resc=5,
                                 psf_fl = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2-60x1p5_final.npy',
                                im_med_fl = r'C:\Scripts\NMERFISH\flat_field\Scope2_60X1p_med_col_raw3.npz'):
    obj = get_dapi_features(fl,save_folder,set_,psf_fl=psf_fl,im_med_fl=im_med_fl)
    obj_ref = get_dapi_features(fl_ref,save_folder_ref,set_,psf_fl=psf_fl,im_med_fl=im_med_fl)
    tzxyf,tzxy_plus,tzxy_minus,N_plus,N_minus = np.array([0,0,0]),np.array([0,0,0]),np.array([0,0,0]),0,0
    if (len(obj.Xh_plus)>0) and (len(obj_ref.Xh_plus)>0):
        X = obj.Xh_plus[:,:3]
        X_ref = obj_ref.Xh_plus[:,:3]
        tzxy_plus,N_plus = get_best_translation_points(X,X_ref,resc=resc,return_counts=True)
    if (len(obj.Xh_minus)>0) and (len(obj_ref.Xh_minus)>0):
        X = obj.Xh_minus[:,:3]
        X_ref = obj_ref.Xh_minus[:,:3]
        tzxy_minus,N_minus = get_best_translation_points(X,X_ref,resc=resc,return_counts=True)
    if np.max(np.abs(tzxy_minus-tzxy_plus))<=2:
        tzxyf = -(tzxy_plus*N_plus+tzxy_minus*N_minus)/(N_plus+N_minus)
    else:
        tzxyf = -[tzxy_plus,tzxy_minus][np.argmax([N_plus,N_minus])]
    return [tzxyf,tzxy_plus,tzxy_minus,N_plus,N_minus]

In [ ]:
all_flds = [r'\\192.168.0.113\Papaya9\Brett\1Hyb_P1F1F2F3']
save_folder = r'\\192.168.0.34\Papaya5\Brett\DNA_analysis'
save_folderf = r'\\192.168.0.34\Papaya5\Brett\ecDNActs'
if not os.path.exists(save_folderf): os.makedirs(save_folderf)

set_ = ''
fov = 'Conv_zscan__014'#Conv_zscan__014
for fov in tqdm(fovs):
    saveflf = save_folderf+os.sep+fov+set_+'_cts_pval_Rg_Rgmn_Rgstd.pkl'
    if not os.path.exists(saveflf):
        dna_folder = r'\\192.168.0.113\Papaya9\Brett\1Hyb_P1F1F2F3'
        fl = dna_folder+os.sep+fov+'.zarr'
        fl_ref = r'\\192.168.0.53\yw1\Brett\Q1_MER'+os.sep+fov+'.zarr'
        save_folder_ref =r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ'
        save_folder_dna =r'\\192.168.0.34\Papaya5\Brett\DNA_analysis'
        drift = get_best_translation_pointsT(fl,save_folder_dna,fl_ref,save_folder_ref,set_='',resc=5,
                                         psf_fl = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2-60x1p5_final.npy',
                                        im_med_fl = r'C:\Scripts\NMERFISH\flat_field\Scope2_60X1p_med_col_raw3.npz')
        drift_change = drift[0]#np.round(drift[0]/resc).astype(int)
        
        Xh1 = np.load(r'\\192.168.0.34\Papaya5\Brett\DNA_analysis'+os.sep+fov+'--1Hyb_P1F1F2F3--col0__Xhfits.npz')['Xh']
        Xh2 = np.load(r'\\192.168.0.34\Papaya5\Brett\DNA_analysis'+os.sep+fov+'--1Hyb_P1F1F2F3--col1__Xhfits.npz')['Xh']
        ths = [np.exp(9.75),np.exp(10.5)]
        Xh1_ = Xh1[Xh1[:,-1]>ths[0]]
        Xh2_ = Xh2[Xh2[:,-1]>ths[1]]
        fovs = list(fovs)
        ifov = fovs.index(fov)
        coords= coordsf[ifov]
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]
        min_ = np.array([0,0,0])+drift_change#np.nanmin(XF[:,:3],axis=0)
        #max_ = np.nanmax(XF[:,:3],axis=0)
        
        xIm,yIm = np.round([xIm,yIm]+(min_/resc)[1:]).astype(int)
        ### deal with the case of outside image
        Im = np.array([xIm,yIm])
        min_[1:]=min_[1:]-Im*(Im<0)*resc[1:]
        ### recenter
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]
        xIm,yIm = np.round([xIm,yIm]+(min_/resc)[1:]).astype(int)
        max_ = np.nanmax(im.shape[-3:],axis=0)
        xIM,yIM = np.round([xIm,yIm]+(max_/resc)[1:]).astype(int)
        xIm=np.max([0,xIm])
        yIm=np.max([0,yIm])
        im_segm_ = np.array(imfsegm[:,xIm:xIM,yIm:yIM])
        
        fl_raw = dna_folder+os.sep+fov+'.zarr'
        im = read_im(fl_raw)
        X1 = Xh1_[:,:3]/resc
        X2 = Xh2_[:,:3]/resc
        X1red = np.round(X1).astype(int)
        X2red = np.round(X2).astype(int)
        kp1 = np.all(X1red<im_segm_.shape,axis=-1)&np.all(X1red>=0,axis=-1)
        X1red = X1red[kp1]
        kp2 = np.all(X2red<im_segm_.shape,axis=-1)&np.all(X2red>=0,axis=-1)
        X2red = X2red[kp2]
        cellsid1 = im_segm_[tuple(X1red.T)]
        cellsid2 = im_segm_[tuple(X2red.T)]
        cells1,cts1 = np.unique(cellsid1,return_counts=True)
        cells1,cts1 = cells1[cells1>0],cts1[cells1>0]
        cells2,cts2 = np.unique(cellsid2,return_counts=True)
        cells2,cts2 = cells2[cells2>0],cts2[cells2>0]
        dic_cts1 = dict(zip(cells1,cts1))
        dic_cts2 = dict(zip(cells2,cts2))
        #x,y = zip(*[(dic_cts1[e],dic_cts2.get(e,0))for e in dic_cts1])
        #plt.plot(np.log1p(x),np.log1p(y),'o')
        if False:
            rz,rx,ry = resc.astype(int)
            imf = im[-1,::rz,::rx,::ry]
            X1 = Xh1_[:,:3]/resc
            X2 = Xh2_[:,:3]/resc
            V = napari.Viewer()
            V.add_image(imf)
            V.add_labels(im_segm_)
            V.add_points(X1)
        def get_cell_info(im_segm_,cell,XT,cellsids,th_cts = 5):
            X_ = XT[cell==cellsids]
            if len(X_)>th_cts:
                Xcell = np.array(np.where(im_segm_ ==cell)).T
                
                def get_rg(X):
                    Xcm = X-np.mean(X,axis=0)
                    return np.median(np.linalg.norm(Xcm,axis=-1))
                Rgrnd = []
                for irnd in range(500):
                    X_rand = Xcell[np.random.choice(np.arange(len(Xcell)),len(X_))]
                    Rgrnd.append(get_rg(X_rand))
                Rgrnd = np.array(Rgrnd)
                Rg=get_rg(X_)
                pval = 1-np.mean(Rg<Rgrnd)
                return len(X_),pval,Rg,np.mean(Rgrnd),np.std(Rgrnd)
            else:
                return len(X_),np.nan,np.nan,np.nan,np.nan
        dic_info1 = {cell:get_cell_info(im_segm_,cell,X1red,cellsid1,th_cts = 5) for cell in cells1}  
        dic_info2 = {cell:get_cell_info(im_segm_,cell,X2red,cellsid2,th_cts = 5) for cell in cells2}  
        pickle.dump([dic_info1,dic_info2],open(saveflf,'wb'))

In [ ]:
fls = glob.glob(r'\\192.168.0.34\Papaya5\Brett\ecDNActs\*_Rgstd.pkl')
dicDNA1 = {}
dicDNA2 = {}
for fl in tqdm(fls):
    dic1,dic2 = pickle.load(open(fl,'rb'))
    dicDNA1.update(dic1)
    dicDNA2.update(dic2)

## add DNA cts and dispersion scores to adata

In [ ]:
import scanpy as sc
data_folder =   r'\\192.168.0.34\Papaya5\Brett\DataFramesV2'
scdata =  sc.read_h5ad(data_folder+os.sep+'scdata_1p5_Brett_6_19_2024.h5ad')

In [ ]:
## DNA cts
cells_scanpy = [int(cell.split('_')[-1])for cell in scdata.obs.index]
dna1_info  = np.array([dicDNA1.get(cell,(0, np.nan,np.nan, np.nan, np.nan))  for cell in cells_scanpy])
cts1 = dna1_info[:,0]
scdata.obs['ecDNA_cts1'] = cts1
dna2_info  = np.array([dicDNA2.get(cell,(0, np.nan,np.nan, np.nan, np.nan))  for cell in cells_scanpy])
cts2 = dna2_info[:,0]
scdata.obs['ecDNA_cts2'] = cts2

In [ ]:
## dispersion score
scdata.obs['RgRatio1']=dna1_info[:,2]/dna1_info[:,3]
scdata.obs['RgRatio2']=dna2_info[:,2]/dna2_info[:,3]

# MALAT

## malat quantification

In [ ]:
save_folder_ref =r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ'

for fov in tqdm(fovs):
    save_folder_malat =r'\\192.168.0.34\Papaya5\Brett\Malat_dics'
    if not os.path.exists(save_folder_malat): os.makedirs(save_folder_malat)
    fl_malat_save = save_folder_malat+os.sep+fov.replace('.zarr','')+'_malatdic.pkl'
    if not os.path.exists(fl_malat_save):
        dec = decoder_simple(save_folder_ref,fov,set_='')
        drifts,flds,fov_,ref_fl = np.load(dec.drift_fl,allow_pickle=True)
        fl_ref = flds[0]+os.sep+fov_
        malat1_folder = r'\\192.168.0.53\yw1\Brett\P_B_Malat1_MALAT1'
        fl = malat1_folder+os.sep+fov_
        
        save_folder_ref =r'\\192.168.0.34\Papaya5\Brett\MERFISH_AnalysisQ'
        drift = get_best_translation_pointsT(fl,save_folder_ref,fl_ref,save_folder_ref,set_='',resc=5,
                                         psf_fl = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2-60x1p5_final.npy',
                                        im_med_fl = r'C:\Scripts\NMERFISH\flat_field\Scope2_60X1p_med_col_raw3.npz')
        drift_change = np.round(drift[0]/resc).astype(int)
        print(drift)
        im = read_im(fl)[[1,2]]
        immeds = [np.load(r'C:\Scripts\NMERFISH\flat_field\Scope2_60X1p_med_col_raw1.npz')['im'],
                  np.load(r'C:\Scripts\NMERFISH\flat_field\Scope2_60X1p_med_col_raw2.npz')['im']]
        imT = []
        for im_,immed in zip(im,immeds):
            immed = cv2.blur(immed,(20,20))
            im__ = np.array(im_,dtype=np.float32)/immed*np.median(immed)
            imT.append(im__)
        im = np.array(imT)
        fovs = list(fovs)
        ifov = fovs.index(dec.fov)
        coords= coordsf[ifov]
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]
        min_ = np.array([0,0,0])-drift_change#np.nanmin(XF[:,:3],axis=0)
        #max_ = np.nanmax(XF[:,:3],axis=0)
        
        xIm,yIm = np.round([xIm,yIm]+(min_/resc)[1:]).astype(int)
        ### deal with the case of outside image
        Im = np.array([xIm,yIm])
        min_[1:]=min_[1:]-Im*(Im<0)*resc[1:]
        ### recenter
        xIm,xIM,yIm,yIM = coords[0],coords[1],coords[2],coords[3]
        xIm,yIm = np.round([xIm,yIm]+(min_/resc)[1:]).astype(int)
        max_ = np.nanmax(im.shape[-3:],axis=0)
        xIM,yIM = np.round([xIm,yIm]+(max_/resc)[1:]).astype(int)
        xIm=np.max([0,xIm])
        yIm=np.max([0,yIm])
        im_segm_ = np.array(imfsegm[:,xIm:xIM,yIm:yIM])
        dec.imfsegm0 = im_segm_.copy()
        print(xIm,xIM,yIm,yIM)
        dec.im_segm_ = im_segm_#expand_segmentation(im_segm_,int(9*1.5))
        rz,rx,ry = resc.astype(int)
        imf = im[...,::rz,::rx,::ry]
        from scipy import ndimage as ndi
        cells = np.unique(dec.im_segm_)
        cells = cells[cells>0]
        signal1 = ndi.mean(imf[0],dec.im_segm_,cells)
        signal2 = ndi.mean(imf[1],dec.im_segm_,cells)
        #plt.scatter(signal1,signal2,s=1)
        dic_malat = {c:[s1,s2]for c,s1,s2 in zip(cells,signal1,signal2)}
        print("Saved:", fl_malat_save)
        pickle.dump(dic_malat,open(fl_malat_save,'wb'))

## add MALAT into the adata

In [ ]:
malat_dic = {}
malat_fls = glob.glob(r'\\192.168.0.34\Papaya5\Brett\Malat_dics\*_malatdic.pkl')
len(malat_fls)
for malat_fl in malat_fls:
    malat_dic.update(pickle.load(open(malat_fl,'rb')))

In [ ]:
cells_scanpy = [int(cell.split('_')[-1])for cell in scdata.obs.index]
malat_info  = [malat_dic.get(cell,[np.nan,np.nan])  for cell in cells_scanpy]
malat_info = np.array(malat_info)

In [ ]:
malat_info = np.array([scdata.obs['Malat1'],scdata.obs['MALAT1']]).T
scdata.obs['Malat1']=malat_info[:,0]
scdata.obs['MALAT1']=malat_info[:,1]

In [ ]:
scdata.write_h5ad(data_folder+os.sep+'scdata_1p5_Brett_6_19_2024_withecDNA_and_malat.h5ad')